# Modern LLMs: APIs, Evaluation & Practical Applications

In this notebook, we explore **modern large language models** (LLMs) like GPT-4, Claude, and others through their APIs. We'll learn practical skills for working with these models in production, including prompt engineering, few-shot learning, and systematic evaluation.

## Learning Objectives

1. Understand the modern LLM landscape (GPT-4, Claude, LLaMA, etc.)
2. Learn to work with LLM APIs (OpenAI, Anthropic)
3. Master prompt engineering techniques
4. Implement few-shot learning
5. Evaluate LLMs systematically (cost vs performance)
6. Understand when to use fine-tuned BERT vs API-based LLMs

---

## 1. The Modern LLM Landscape

### Major Models & Providers

| Model Family | Provider | Size Range | Key Features | API Access |
|--------------|----------|------------|--------------|------------|
| **GPT-4** | OpenAI | ~1.7T params | Most capable, multimodal | ✅ Paid |
| **GPT-3.5** | OpenAI | 175B params | Fast, cost-effective | ✅ Paid |
| **Claude** | Anthropic | ~100B-200B | Long context (200K), safety-focused | ✅ Paid |
| **Gemini** | Google | Various | Multimodal, integrated with Google services | ✅ Paid |
| **LLaMA** | Meta | 7B-70B | Open weights, self-hostable | ✅ Free (self-host) |
| **Mistral** | Mistral AI | 7B-8x7B (MoE) | Efficient, open weights | ✅ Free/Paid |
| **BERT** | Google | 110M-340M | Fine-tuning required, task-specific | ✅ Free (self-host) |

### Key Capabilities Evolution

```
BERT (2018)          →  GPT-3 (2020)       →  GPT-4 (2023)
────────────────────────────────────────────────────────────
• Fine-tune required    • Zero/few-shot       • Multimodal
• Task-specific         • Prompt-based        • 128K context
• 512 tokens max        • 4K-32K tokens       • Better reasoning
• No generation         • Text generation     • Tool use
• Cheap to run          • Moderate cost       • Higher cost
```

### When to Use What?

**Use Fine-tuned BERT when:**
- You have labeled training data
- Low latency required (<10ms)
- Predictable costs important
- Task is classification/NER/Q&A extraction
- Need to run offline/on-device

**Use API-based LLMs when:**
- No training data available
- Need generation/creative tasks
- Requirements change frequently
- Can tolerate higher latency (100ms-1s)
- Need reasoning/complex instructions

---
## 2. Setup & API Configuration

**⚠️ Important:** This notebook demonstrates API usage patterns. To run these examples, you'll need:
- OpenAI API key (from platform.openai.com)
- Anthropic API key (from console.anthropic.com)

For educational purposes, you can also use mock responses or local models.

In [ ]:
# Install required packages
!pip install -q openai anthropic tiktoken pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import time
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For token counting
import tiktoken

# Set your API keys (use environment variables in production!)
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

# For demonstration, we'll use mock mode
USE_MOCK_RESPONSES = True  # Set to False if you have API keys

print("Setup complete!")
if USE_MOCK_RESPONSES:
    print("⚠️ Running in MOCK mode (no real API calls)")
else:
    print("✅ Running with real API calls")

---
## 3. Working with OpenAI API (GPT-4 / GPT-3.5)

### 3.1 Basic Chat Completion

In [ ]:
def call_openai(prompt: str, model: str = "gpt-3.5-turbo", temperature: float = 0.7, max_tokens: int = 150) -> Dict:
    """
    Call OpenAI API with error handling and metrics.
    
    Args:
        prompt: The user prompt
        model: Model to use (gpt-3.5-turbo, gpt-4, etc.)
        temperature: Sampling temperature
        max_tokens: Maximum tokens to generate
    
    Returns:
        Dictionary with response, usage stats, and metrics
    """
    if USE_MOCK_RESPONSES:
        # Mock response for demonstration
        time.sleep(0.5)  # Simulate API latency
        return {
            'response': f"[Mock Response] This is a simulated answer to: '{prompt[:50]}...'",
            'model': model,
            'input_tokens': len(prompt.split()) * 1.3,  # Approximate
            'output_tokens': 30,
            'total_tokens': len(prompt.split()) * 1.3 + 30,
            'latency': 0.5,
            'cost': 0.001  # Approximate
        }
    
    # Real API call
    from openai import OpenAI
    client = OpenAI()
    
    start_time = time.time()
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    
    latency = time.time() - start_time
    
    # Calculate cost (approximate, as of 2024)
    cost_per_1k_input = 0.5 if model == "gpt-3.5-turbo" else 5.0  # USD
    cost_per_1k_output = 1.5 if model == "gpt-3.5-turbo" else 15.0
    
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (input_tokens / 1000 * cost_per_1k_input) + (output_tokens / 1000 * cost_per_1k_output)
    
    return {
        'response': response.choices[0].message.content,
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': response.usage.total_tokens,
        'latency': latency,
        'cost': cost
    }

In [ ]:
# Test basic completion
prompt = "Explain what a transformer is in machine learning in 2 sentences."
result = call_openai(prompt, model="gpt-3.5-turbo")

print(f"Prompt: {prompt}\n")
print(f"Response: {result['response']}\n")
print(f"Model: {result['model']}")
print(f"Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total")
print(f"Latency: {result['latency']:.3f}s")
print(f"Cost: ${result['cost']:.6f}")

---
## 4. Prompt Engineering

The quality of LLM outputs depends heavily on **how you ask**.

### 4.1 Basic Principles

1. **Be specific**: Clear instructions yield better results
2. **Provide context**: Give background information
3. **Show examples**: Few-shot learning improves accuracy
4. **Specify format**: Tell the model how to structure output
5. **Iterate**: Refine prompts based on results

In [ ]:
# Compare bad vs good prompts
prompts_comparison = [
    {
        'name': 'Bad: Vague',
        'prompt': 'Tell me about AI'
    },
    {
        'name': 'Better: Specific',
        'prompt': 'Explain what artificial intelligence is and list 3 main subfields, in 3 sentences.'
    },
    {
        'name': 'Best: Specific + Format',
        'prompt': '''Explain artificial intelligence.

Format your answer as:
1. Definition (1 sentence)
2. Three main subfields (bullet points)
3. One real-world application (1 sentence)'''
    }
]

print("Comparing Prompt Quality:\n")
print("="*80)

for item in prompts_comparison:
    print(f"\n{item['name']}:")
    print(f"Prompt: {item['prompt']}")
    result = call_openai(item['prompt'], max_tokens=200)
    print(f"\nResponse: {result['response']}")
    print("\n" + "="*80)

### 4.2 Few-Shot Learning

Provide examples in the prompt to teach the model the task pattern.

In [ ]:
# Zero-shot vs Few-shot comparison
task = "sentiment classification"

# Zero-shot
zero_shot_prompt = """Classify the sentiment of this text as positive, negative, or neutral.

Text: "This product is amazing!"
Sentiment:"""

# Few-shot
few_shot_prompt = """Classify the sentiment of each text as positive, negative, or neutral.

Text: "I love this product! It exceeded my expectations."
Sentiment: positive

Text: "Terrible quality. Waste of money."
Sentiment: negative

Text: "It's okay, nothing special."
Sentiment: neutral

Text: "This product is amazing!"
Sentiment:"""

print("Zero-shot vs Few-shot Learning:\n")
print("="*80)

print("\nZero-shot:")
result_zero = call_openai(zero_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_zero['response']}")

print("\nFew-shot:")
result_few = call_openai(few_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_few['response']}")

print("\n" + "="*80)
print("\n✅ Few-shot typically gives more consistent, accurate results")

### 4.3 Chain-of-Thought Prompting

Ask the model to "think step by step" for better reasoning.

In [ ]:
# Without CoT
direct_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon. 
Then they sell 12 more. How many apples do they have?

A:"""

# With Chain-of-Thought
cot_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon. 
Then they sell 12 more. How many apples do they have?

A: Let's solve this step by step:"""

print("Chain-of-Thought Prompting:\n")
print("="*80)

print("\nDirect:")
result_direct = call_openai(direct_prompt, temperature=0.0, max_tokens=50)
print(result_direct['response'])

print("\n" + "="*80)
print("\nChain-of-Thought:")
result_cot = call_openai(cot_prompt, temperature=0.0, max_tokens=100)
print(result_cot['response'])

print("\n" + "="*80)
print("\n✅ CoT helps LLMs show their reasoning and catch errors")

---
## 5. LLM Evaluation Framework

How do we systematically evaluate LLMs?

### 5.1 Task-Specific Evaluation

For tasks with ground truth, we can measure accuracy.

In [ ]:
# Create evaluation dataset
eval_dataset = [
    {
        'text': 'I absolutely loved this movie! Best film of the year.',
        'true_sentiment': 'positive'
    },
    {
        'text': 'Terrible service. Never coming back.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'The product works as expected.',
        'true_sentiment': 'neutral'
    },
    {
        'text': 'Waste of time and money. Very disappointed.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'Excellent quality! Highly recommend.',
        'true_sentiment': 'positive'
    }
]

def evaluate_sentiment_model(dataset: List[Dict], model: str = "gpt-3.5-turbo") -> Dict:
    """
    Evaluate LLM on sentiment classification task.
    """
    predictions = []
    total_cost = 0
    total_latency = 0
    
    for item in dataset:
        prompt = f"""Classify sentiment as: positive, negative, or neutral.

Text: "{item['text']}"
Sentiment:"""
        
        result = call_openai(prompt, model=model, temperature=0.0, max_tokens=10)
        pred = result['response'].strip().lower()
        
        # Extract sentiment label
        if 'positive' in pred:
            pred_label = 'positive'
        elif 'negative' in pred:
            pred_label = 'negative'
        else:
            pred_label = 'neutral'
        
        predictions.append(pred_label)
        total_cost += result['cost']
        total_latency += result['latency']
    
    # Calculate accuracy
    true_labels = [item['true_sentiment'] for item in dataset]
    accuracy = sum([p == t for p, t in zip(predictions, true_labels)]) / len(dataset)
    
    return {
        'model': model,
        'accuracy': accuracy,
        'total_cost': total_cost,
        'avg_latency': total_latency / len(dataset),
        'predictions': predictions,
        'true_labels': true_labels
    }

# Evaluate
results = evaluate_sentiment_model(eval_dataset, model="gpt-3.5-turbo")

print("Sentiment Classification Evaluation:\n")
print(f"Model: {results['model']}")
print(f"Accuracy: {results['accuracy']:.2%}")
print(f"Total Cost: ${results['total_cost']:.4f}")
print(f"Avg Latency: {results['avg_latency']:.3f}s")
print("\nPredictions vs True Labels:")
for i, (pred, true) in enumerate(zip(results['predictions'], results['true_labels'])):
    match = "✓" if pred == true else "✗"
    print(f"  {i+1}. {match} Predicted: {pred:8} | True: {true}")

### 5.2 Model Comparison: GPT-3.5 vs GPT-4 vs Fine-tuned BERT

In [ ]:
# Simulate comparison (use real results if you have API access)
comparison_data = [
    {
        'Model': 'Fine-tuned BERT',
        'Accuracy': 0.94,
        'Avg Latency (ms)': 15,
        'Cost per 1K queries': 0.10,  # Compute cost
        'Setup Cost': 'High (training)',
        'Update Cost': 'High (retrain)'
    },
    {
        'Model': 'GPT-3.5 (zero-shot)',
        'Accuracy': 0.85,
        'Avg Latency (ms)': 450,
        'Cost per 1K queries': 0.50,
        'Setup Cost': 'Low (prompt)',
        'Update Cost': 'Low (edit prompt)'
    },
    {
        'Model': 'GPT-3.5 (few-shot)',
        'Accuracy': 0.92,
        'Avg Latency (ms)': 500,
        'Cost per 1K queries': 0.75,
        'Setup Cost': 'Low (prompt)',
        'Update Cost': 'Low (edit prompt)'
    },
    {
        'Model': 'GPT-4 (zero-shot)',
        'Accuracy': 0.96,
        'Avg Latency (ms)': 800,
        'Cost per 1K queries': 5.00,
        'Setup Cost': 'Low (prompt)',
        'Update Cost': 'Low (edit prompt)'
    }
]

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("MODEL COMPARISON: Sentiment Classification Task")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

In [ ]:
# Visualize trade-offs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy vs Cost
axes[0].scatter(df_comparison['Cost per 1K queries'], 
                df_comparison['Accuracy'], 
                s=200, alpha=0.6, c=range(len(df_comparison)), cmap='viridis')

for i, row in df_comparison.iterrows():
    axes[0].annotate(row['Model'], 
                     (row['Cost per 1K queries'], row['Accuracy']),
                     fontsize=9, ha='right')

axes[0].set_xlabel('Cost per 1K Queries ($)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Cost Trade-off', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_xscale('log')

# Plot 2: Latency comparison
axes[1].barh(df_comparison['Model'], df_comparison['Avg Latency (ms)'], 
             color='skyblue', edgecolor='black')
axes[1].set_xlabel('Average Latency (ms)', fontsize=12)
axes[1].set_title('Inference Latency Comparison', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Key Insights:")
print("  • Fine-tuned BERT: Best latency, low cost, but requires training")
print("  • GPT-3.5: Good balance for prototyping and changing requirements")
print("  • GPT-4: Best accuracy but highest cost - use for critical tasks")
print("  • Few-shot improves accuracy significantly over zero-shot")

### 5.3 LLM-as-a-Judge

For tasks without clear ground truth (e.g., creative writing, summarization), use another LLM to evaluate.

In [ ]:
def llm_judge(text: str, criteria: List[str]) -> Dict[str, float]:
    """
    Use LLM to evaluate text on multiple criteria.
    
    Args:
        text: Text to evaluate
        criteria: List of evaluation criteria
    
    Returns:
        Dictionary of scores (1-5) for each criterion
    """
    criteria_str = "\n".join([f"- {c}" for c in criteria])
    
    prompt = f"""Evaluate the following text on these criteria (rate each 1-5, where 5 is best):

{criteria_str}

Text: "{text}"

Provide your ratings in this format:
Criterion: score
(one per line)"""
    
    result = call_openai(prompt, temperature=0.0, max_tokens=100)
    
    # Parse scores (simplified)
    scores = {}
    for criterion in criteria:
        # In mock mode, assign random scores
        if USE_MOCK_RESPONSES:
            scores[criterion] = np.random.randint(3, 6)
        else:
            # Parse from response (you'd need more robust parsing in production)
            scores[criterion] = 4  # Placeholder
    
    return scores

# Example evaluation
generated_summary = "Machine learning enables computers to learn from data and improve over time without explicit programming."

evaluation_criteria = [
    "Accuracy (factually correct)",
    "Clarity (easy to understand)",
    "Conciseness (not verbose)",
    "Completeness (covers key points)"
]

scores = llm_judge(generated_summary, evaluation_criteria)

print("LLM-as-Judge Evaluation:\n")
print(f"Text: {generated_summary}\n")
print("Scores (1-5):")
for criterion, score in scores.items():
    print(f"  • {criterion}: {score}/5")
print(f"\nAverage Score: {np.mean(list(scores.values())):.2f}/5")

---
## 6. Cost-Performance Optimization

In production, you need to balance quality, cost, and latency.

### 6.1 Cost Projection

In [ ]:
# Simulate cost at different scales
def calculate_monthly_cost(queries_per_day: int, cost_per_query: float) -> float:
    return queries_per_day * 30 * cost_per_query

query_volumes = [100, 1000, 10000, 100000]
cost_scenarios = []

for volume in query_volumes:
    cost_scenarios.append({
        'Queries/Day': f"{volume:,}",
        'BERT (fine-tuned)': f"${calculate_monthly_cost(volume, 0.0001):.2f}",
        'GPT-3.5': f"${calculate_monthly_cost(volume, 0.0005):.2f}",
        'GPT-4': f"${calculate_monthly_cost(volume, 0.005):.2f}"
    })

df_cost = pd.DataFrame(cost_scenarios)

print("\n💰 Monthly Cost Projections\n")
print(df_cost.to_string(index=False))
print("\n⚠️ At scale, costs become significant! Optimize wisely.")

### 6.2 Optimization Strategies

In [ ]:
optimization_strategies = pd.DataFrame([
    {
        'Strategy': 'Use cheaper model (GPT-3.5 vs GPT-4)',
        'Cost Reduction': '90%',
        'Quality Impact': 'Moderate',
        'Effort': 'Low'
    },
    {
        'Strategy': 'Reduce output tokens (shorter answers)',
        'Cost Reduction': '30-50%',
        'Quality Impact': 'Low',
        'Effort': 'Low'
    },
    {
        'Strategy': 'Cache frequent queries',
        'Cost Reduction': '50-80%',
        'Quality Impact': 'None',
        'Effort': 'Medium'
    },
    {
        'Strategy': 'Route simple queries to smaller model',
        'Cost Reduction': '40-60%',
        'Quality Impact': 'Low',
        'Effort': 'High'
    },
    {
        'Strategy': 'Fine-tune small model for specific task',
        'Cost Reduction': '95%',
        'Quality Impact': 'Varies',
        'Effort': 'Very High'
    },
    {
        'Strategy': 'Batch processing (delay tolerance)',
        'Cost Reduction': '50%',
        'Quality Impact': 'None',
        'Effort': 'Medium'
    }
])

print("\n🎯 Cost Optimization Strategies\n")
print(optimization_strategies.to_string(index=False))

---
## 7. Decision Framework: When to Use What?

A practical guide for choosing the right approach.

In [ ]:
decision_tree = """
┌─ Do you have labeled training data (>1000 examples)?
│
├─ YES → Consider Fine-tuning
│  ├─ Need very low latency (<50ms)? → Fine-tune BERT
│  ├─ Need generation capability? → Fine-tune GPT/T5
│  └─ Requirements change often? → Maybe use API instead
│
└─ NO → Use API-based LLMs
   ├─ Budget constrained? → GPT-3.5 or open-source (LLaMA, Mistral)
   ├─ Need best quality? → GPT-4 or Claude
   ├─ Need long context (>32K)? → Claude (200K), GPT-4-turbo (128K)
   └─ Privacy-sensitive data? → Self-host open model (LLaMA 2, Mistral)

Special Considerations:
• High volume (>100K queries/day)? → Fine-tuning becomes cost-effective
• Rapidly evolving domain? → RAG + API for flexibility
• Need citations/explainability? → RAG is essential
• Offline/embedded use? → Must fine-tune or use small models
"""

print(decision_tree)

---
## 8. Production Best Practices

### 8.1 Error Handling & Retries

In [ ]:
def call_llm_with_retry(prompt: str, max_retries: int = 3) -> Dict:
    """
    Call LLM API with exponential backoff retry logic.
    """
    for attempt in range(max_retries):
        try:
            result = call_openai(prompt)
            return result
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait_time = 2 ** attempt  # Exponential backoff
            print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

print("✅ Always implement retry logic with exponential backoff")
print("✅ Handle rate limits gracefully")
print("✅ Log all API calls for debugging and cost tracking")

### 8.2 Monitoring & Alerts

In [ ]:
monitoring_checklist = [
    "✓ Track latency (p50, p95, p99)",
    "✓ Monitor error rates",
    "✓ Track daily/monthly costs",
    "✓ Measure quality metrics (accuracy, user feedback)",
    "✓ Alert on anomalies (sudden cost spike, latency increase)",
    "✓ Log all prompts and responses (for debugging)",
    "✓ A/B test different prompts/models",
    "✓ Collect user feedback (thumbs up/down)"
]

print("📊 Production Monitoring Checklist:\n")
for item in monitoring_checklist:
    print(f"  {item}")

### 8.3 Safety & Content Filtering

In [ ]:
safety_considerations = pd.DataFrame([
    {'Risk': 'Harmful content generation', 'Mitigation': 'Use built-in content filters, add guardrails'},
    {'Risk': 'PII leakage', 'Mitigation': 'Sanitize inputs, don\'t log sensitive data'},
    {'Risk': 'Prompt injection attacks', 'Mitigation': 'Validate inputs, use system prompts'},
    {'Risk': 'Biased outputs', 'Mitigation': 'Test across demographics, use diverse examples'},
    {'Risk': 'Hallucination', 'Mitigation': 'Use RAG, fact-check critical information'},
    {'Risk': 'Cost overruns', 'Mitigation': 'Set rate limits, budget alerts, caching'}
])

print("\n🔒 Safety & Risk Mitigation\n")
print(safety_considerations.to_string(index=False))

---
## 9. Benchmark Suites (Reference)

Standard benchmarks for evaluating LLMs:

| Benchmark | What it Measures | Example Tasks |
|-----------|------------------|---------------|
| **MMLU** | Multitask knowledge | History, physics, law questions |
| **HellaSwag** | Commonsense reasoning | Complete the scenario |
| **GSM8K** | Math reasoning | Grade school math problems |
| **HumanEval** | Code generation | Python function completion |
| **TruthfulQA** | Truthfulness | Avoiding false statements |
| **BBH (Big-Bench Hard)** | Challenging reasoning | Logic, planning, deduction |

### Current State-of-the-Art (approximate, as of 2024)

- **GPT-4**: MMLU 86%, GSM8K 92%, HumanEval 67%
- **Claude 3 Opus**: MMLU 86%, GSM8K 95%, HumanEval 84%
- **GPT-3.5**: MMLU 70%, GSM8K 57%, HumanEval 48%
- **LLaMA 2 70B**: MMLU 69%, GSM8K 56%, HumanEval 29%

**Note**: These are reference numbers. Always evaluate on *your specific task*!

---
## Summary

### Key Takeaways

1. **Modern LLMs are powerful but expensive**
   - GPT-4: Best quality, highest cost
   - GPT-3.5: Good balance for most tasks
   - Fine-tuned BERT: Best for high-volume, low-latency tasks

2. **Prompt engineering is critical**
   - Be specific and provide context
   - Use few-shot learning for consistency
   - Chain-of-thought for complex reasoning

3. **Evaluation is multi-dimensional**
   - Accuracy (task-specific metrics)
   - Latency (user experience)
   - Cost (operational feasibility)
   - Quality (LLM-as-judge for subjective tasks)

4. **Optimization strategies**
   - Cache frequent queries
   - Route intelligently (simple → cheap, complex → expensive)
   - Use cheaper models when possible
   - Fine-tune for high-volume use cases

5. **Production considerations**
   - Implement retry logic and error handling
   - Monitor costs, latency, and quality
   - Add safety guardrails
   - Collect user feedback

### Decision Framework Summary

```
Have labeled data + high volume → Fine-tune BERT/small model
Prototyping / changing requirements → GPT-3.5 API
Critical quality / complex reasoning → GPT-4 API
Need external knowledge → RAG + LLM
Privacy-sensitive → Self-host open model
```

### Connection to Previous Notebooks

We've now completed the journey:
1. **BERT (12)**: Fine-tuned encoders for understanding
2. **GPT (13)**: Decoders for generation
3. **RAG (14-1)**: Combining retrieval + generation
4. **Modern LLMs (14-2)**: Production-ready API usage

You now have the tools to build state-of-the-art NLP systems! 🎉

---

### Further Reading

- [OpenAI API Documentation](https://platform.openai.com/docs/)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [LLM Leaderboards (HuggingFace)](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)
- [Papers: Chain-of-Thought Prompting (Wei et al., 2022)](https://arxiv.org/abs/2201.11903)